In [1]:
'''import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    recall_score, precision_score, confusion_matrix
)'''

"import warnings\nwarnings.filterwarnings('ignore')\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.metrics import (\n    accuracy_score, roc_auc_score, f1_score,\n    recall_score, precision_score, confusion_matrix\n)"

In [2]:
'''

# 1. Semilla de reproducibilidad
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# 2. Carga y estructuración de dataset.csv (Formato Melt -> Matrices del Grafo)
df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep=None, engine= 'python')
# Mapeo ordenado de las 153 bacterias y los 121 pacientes
bact_map = df[['bacteria_id', 'Bacteria_Limpia']].drop_duplicates().sort_values('bacteria_id')
nombres_bacterias = bact_map['Bacteria_Limpia'].values

pat_map = df[['paciente_id', 'Paciente', 'Diagnosticos']].drop_duplicates().sort_values('paciente_id')
y_all = pat_map['Diagnosticos'].values  # 1 = TEA (71 pacientes), 0 = HC (50 pacientes)

# Matriz de Abundancia CLR (121 x 153) y Máscara de Presencia de Aristas Bipartitas
pivot_clr = df.pivot(index='paciente_id', columns='bacteria_id', values='AbundanciaTaxonomica')
X_all = pivot_clr.fillna(0.0).values
M_all = (~pivot_clr.isna()).astype(np.float32).values


# 3. Capa GAT Relacional para un color de cable Bacteria <-> Bacteria
class RelationalBacteriaGATLayer(nn.Module):
    """
    Aplica Graph Attention sobre un tipo específico de cable Bacteria <-> Bacteria
    (Rojo = Red TEA, Verde = Red HC, Violeta = Recableado Diferencial).
    """
    def __init__(self, in_features, out_features, dropout=0.25, alpha=0.2):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a_src = nn.Parameter(torch.empty(size=(out_features, 1)))
        self.a_dst = nn.Parameter(torch.empty(size=(out_features, 1)))
        self.a_edge = nn.Parameter(torch.empty(size=(1, 1)))

        nn.init.xavier_uniform_(self.W.weight, gain=1.414)
        nn.init.xavier_uniform_(self.a_src.data, gain=1.414)
        nn.init.xavier_uniform_(self.a_dst.data, gain=1.414)
        nn.init.constant_(self.a_edge.data, 1.0)

        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h_bact, adj_matrix):
        Wh = self.W(h_bact)  # (153, out_features)
        e_src = torch.matmul(Wh, self.a_src)                  # (153, 1)
        e_dst = torch.matmul(Wh, self.a_dst).transpose(0, 1)  # (1, 153)

        # Atención modulada por la intensidad de correlación del cable
        e = self.leakyrelu(e_src + e_dst + self.a_edge * torch.abs(adj_matrix))

        eye = torch.eye(adj_matrix.size(0), device=adj_matrix.device)
        mask = (torch.abs(adj_matrix) > 0) | (eye > 0)
        e_masked = e.masked_fill(~mask, -1e9)

        attention = F.softmax(e_masked, dim=1)
        attention_drop = self.dropout(attention)

        # Propagación respetando el signo de la correlación (simbiosis vs competencia)
        signed_weights = adj_matrix + eye
        h_prime = torch.matmul(attention_drop * signed_weights, Wh)
        return F.elu(h_prime), attention


# 4. Arquitectura GAT Heterogénea Multirelacional (4 Colores de Cables)
class MultiColorHeteroGAT(nn.Module):
    def __init__(self, bact_in_dim=9, hidden_dim=32, dropout=0.25):
        super().__init__()
        # Fase 1: Cabezales GAT independientes por cada color de cable Bacteria-Bacteria
        self.gat_red = RelationalBacteriaGATLayer(bact_in_dim, hidden_dim, dropout=dropout)    # Cable Rojo (TEA)
        self.gat_green = RelationalBacteriaGATLayer(bact_in_dim, hidden_dim, dropout=dropout)  # Cable Verde (HC)
        self.gat_purple = RelationalBacteriaGATLayer(bact_in_dim, hidden_dim, dropout=dropout) # Cable Violeta (Delta)
        self.bact_self = nn.Linear(bact_in_dim, hidden_dim)

        # Integración multicanal de cada bacteria: [Propia, Roja, Verde, Violeta, Roja - Verde]
        self.bact_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 5, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU()
        )

        # Fase 2: Atención Dirigida Bacteria -> Paciente (Cable Azul de Abundancia CLR)
        self.att_bact = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.att_edge = nn.Linear(3, hidden_dim, bias=False)  # [CLR_norm, Z_score_vs_HC, Presencia]
        self.att_vec = nn.Linear(hidden_dim, 1, bias=False)

        self.msg_proj = nn.Linear(hidden_dim + 3, hidden_dim)

        # Clasificador final de Nodos Paciente (TEA vs HC)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 16),
            nn.ELU(),
            nn.Linear(16, 1)
        )

    def forward(self, bact_features, adj_red, adj_green, adj_purple, edge_feats_pb):
        # 1. Aprendizaje Ecológico Bacteria <-> Bacteria
        h_self = F.elu(self.bact_self(bact_features))
        h_red, att_red = self.gat_red(bact_features, adj_red)
        h_green, att_green = self.gat_green(bact_features, adj_green)
        h_purple, att_purple = self.gat_purple(bact_features, adj_purple)
        h_diff = h_red - h_green  # Contraste directo entre cable rojo y cable verde

        h_bact = self.bact_fusion(
            torch.cat([h_self, h_red, h_green, h_purple, h_diff], dim=-1)
        )  # (153, H)

        # 2. Paso de Mensajes Dirigido Bacteria -> Paciente (Bloquea Paciente -> Bacteria -> Paciente)
        P, B, _ = edge_feats_pb.shape
        h_bact_exp = h_bact.unsqueeze(0).expand(P, B, -1)  # (121, 153, H)

        e_pb = F.leaky_relu(self.att_bact(h_bact_exp) + self.att_edge(edge_feats_pb), negative_slope=0.2)
        scores_pb = self.att_vec(e_pb).squeeze(-1)         # (121, 153)

        alpha_pb = F.softmax(scores_pb, dim=1)             # Atención de cada paciente hacia las 153 bacterias

        clr_norm = edge_feats_pb[:, :, 0:1]
        msg_input = torch.cat([h_bact_exp * clr_norm, edge_feats_pb], dim=-1)
        messages = self.msg_proj(msg_input)                # (121, 153, H)

        # Agregación en cada nodo Paciente
        h_patient = torch.sum(alpha_pb.unsqueeze(-1) * messages, dim=1)  # (121, H)
        logits = self.classifier(h_patient).squeeze(-1)

        return logits, alpha_pb, att_red, att_green, h_diff


# 5. Constructor del Grafo dentro de cada Fold (CERO DATA LEAKAGE)
def build_fold_hetero_graph(X_all, M_all, y_all, train_idx, corr_thresh=0.35, diff_thresh=0.40):
    X_train = X_all[train_idx]
    M_train = M_all[train_idx]
    y_train = y_all[train_idx]

    X_train_tea = X_train[y_train == 1]
    X_train_hc = X_train[y_train == 0]

    # Matrices de co-ocurrencia calculadas SOLO en train_idx
    corr_tea = np.nan_to_num(np.corrcoef(X_train_tea, rowvar=False), 0.0)
    corr_hc = np.nan_to_num(np.corrcoef(X_train_hc, rowvar=False), 0.0)
    np.fill_diagonal(corr_tea, 0.0)
    np.fill_diagonal(corr_hc, 0.0)
    corr_diff = corr_tea - corr_hc

    # Cables Rojos (TEA), Verdes (HC) y Violetas (Recableado)
    adj_red = np.where(np.abs(corr_tea) >= corr_thresh, corr_tea, 0.0)
    adj_green = np.where(np.abs(corr_hc) >= corr_thresh, corr_hc, 0.0)
    adj_purple = np.where(np.abs(corr_diff) >= diff_thresh, corr_diff, 0.0)

    # Características del nodo Bacteria (calculadas solo con train_idx)
    mean_tea = X_train_tea.mean(axis=0)
    mean_hc = X_train_hc.mean(axis=0)
    std_tea = X_train_tea.std(axis=0) + 1e-6
    std_hc = X_train_hc.std(axis=0) + 1e-6
    effect_size = (mean_tea - mean_hc) / np.sqrt(0.5 * (std_tea**2 + std_hc**2))

    prev_tea = M_train[y_train == 1].mean(axis=0)
    prev_hc = M_train[y_train == 0].mean(axis=0)
    deg_red = (np.abs(adj_red) > 0).mean(axis=0)
    deg_green = (np.abs(adj_green) > 0).mean(axis=0)
    deg_purple = (np.abs(adj_purple) > 0).mean(axis=0)

    bact_features = np.stack([
        mean_tea, mean_hc, effect_size,
        prev_tea, prev_hc, prev_tea - prev_hc,
        deg_red, deg_green, deg_purple
    ], axis=1)

    # Cables Azules Bacteria -> Paciente (estandarizados con parámetros de train_idx)
    mean_global = X_train.mean(axis=0, keepdims=True)
    std_global = X_train.std(axis=0, keepdims=True) + 1e-6
    X_norm = (X_all - mean_global) / std_global
    z_from_hc = (X_all - mean_hc[None, :]) / std_hc[None, :]

    edge_feats_pb = np.stack([X_norm, z_from_hc, M_all], axis=-1)

    return (
        torch.tensor(bact_features, dtype=torch.float32),
        torch.tensor(adj_red, dtype=torch.float32),
        torch.tensor(adj_green, dtype=torch.float32),
        torch.tensor(adj_purple, dtype=torch.float32),
        torch.tensor(edge_feats_pb, dtype=torch.float32)
    )


# 6. Ejecución de Validación Cruzada Estratificada (5-Fold CV) y Extracción de Biomarcadores TEA
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas_folds = []

# Acumuladores de explicabilidad para niños con TEA
atencion_tea_acum = np.zeros(len(nombres_bacterias))
recableado_acum = np.zeros(len(nombres_bacterias))
gradiente_tea_acum = np.zeros(len(nombres_bacterias))

print("=== ENTRENANDO GAT HETEROGÉNEO MULTIRELACIONAL (5-FOLD CV) ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    set_seed(42 + fold)
    bact_feat, adj_red, adj_green, adj_purple, edge_pb = build_fold_hetero_graph(
        X_all, M_all, y_all, train_idx, corr_thresh=0.35, diff_thresh=0.40
    )
    y_tensor = torch.tensor(y_all, dtype=torch.float32)

    model = MultiColorHeteroGAT(bact_in_dim=9, hidden_dim=32, dropout=0.25)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.006, weight_decay=1.5e-2)

    pos_weight = torch.tensor([(y_all[train_idx] == 0).sum() / (y_all[train_idx] == 1).sum()], dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    for epoch in range(110):
        model.train()
        optimizer.zero_grad()
        logits, _, _, _, _ = model(bact_feat, adj_red, adj_green, adj_purple, edge_pb)
        loss = criterion(logits[train_idx], y_tensor[train_idx])
        loss.backward()
        optimizer.step()

    # Evaluación en el conjunto de validación (val_idx)
    model.eval()
    edge_pb_grad = edge_pb.clone().detach().requires_grad_(True)
    logits, alpha_pb, att_red, att_green, h_diff = model(bact_feat, adj_red, adj_green, adj_purple, edge_pb_grad)

    # Gradiente de atribución hacia el diagnóstico positivo de TEA (Saliency por bacteria)
    tea_val_nodes = val_idx[y_all[val_idx] == 1]
    score_tea = logits[tea_val_nodes].sum()
    score_tea.backward()

    with torch.no_grad():
        probs = torch.sigmoid(logits[val_idx]).numpy()
        preds = (probs >= 0.5).astype(int)
        y_val = y_all[val_idx]

        acc = accuracy_score(y_val, preds)
        auc = roc_auc_score(y_val, probs)
        f1 = f1_score(y_val, preds)
        sens = recall_score(y_val, preds)
        tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
        spec = tn / (tn + fp)

        metricas_folds.append([fold + 1, acc, auc, f1, sens, spec])
        print(f"Fold {fold+1} | Acc: {acc:.4f} | ROC-AUC: {auc:.4f} | F1: {f1:.4f} | Sens(TEA): {sens:.4f} | Spec(HC): {spec:.4f}")

        # 1. Atención asignada por el GAT específicamente en los niños con TEA
        atencion_tea_acum += alpha_pb[tea_val_nodes].mean(dim=0).numpy() / 5.0
        # 2. Magnitud del recableado entre el Cable Rojo y el Cable Verde
        recableado_acum += torch.norm(h_diff, p=2, dim=1).numpy() / 5.0
        # 3. Impacto directo del cable sobre la predicción de TEA
        grad_bact = edge_pb_grad.grad[tea_val_nodes, :, 0].abs().mean(dim=0).numpy()
        gradiente_tea_acum += grad_bact / 5.0

df_metricas = pd.DataFrame(metricas_folds, columns=['Fold', 'Accuracy', 'ROC_AUC', 'F1_Score', 'Sensibilidad_TEA', 'Especificidad_HC'])
print("\n=== RESUMEN DE VALIDACIÓN CRUZADA ===")
print(df_metricas.mean().to_string())

# 7. Construcción del Ranking Final de Bacterias Críticas para Niños con TEA
mean_clr_tea = X_all[y_all == 1].mean(axis=0)
mean_clr_hc = X_all[y_all == 0].mean(axis=0)
delta_clr = mean_clr_tea - mean_clr_hc

# Normalización Min-Max para combinar Atención + Recableado (Rojo vs Verde) + Gradiente TEA
def norm_minmax(v):
    return (v - v.min()) / (v.max() - v.min() + 1e-8)

score_integral_tea = (
    0.45 * norm_minmax(atencion_tea_acum) +
    0.35 * norm_minmax(gradiente_tea_acum) +
    0.20 * norm_minmax(recableado_acum)
)

df_biomarcadores_tea = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Score_GAT_TEA': score_integral_tea,
    'Atencion_Pacientes_TEA': atencion_tea_acum,
    'Indice_Recableado_Rojo_Verde': recableado_acum,
    'Media_CLR_TEA': mean_clr_tea,
    'Media_CLR_HC': mean_clr_hc,
    'Delta_CLR (TEA-HC)': delta_clr,
    'Perfil_en_TEA': np.where(delta_clr > 0, 'Aumentada en TEA (Cable Rojo/Sobrecrecimiento)', 'Disminuida en TEA (Pérdida de Cable Verde)')
}).sort_values(by='Score_GAT_TEA', ascending=False)

print("\n=== TOP 20 BACTERIAS MÁS IMPORTANTES PARA NIÑOS CON TEA (GAT HETEROGÉNEO) ===")
print(df_biomarcadores_tea.head(20).to_string(index=False))'''

'\n\n# 1. Semilla de reproducibilidad\ndef set_seed(seed=42):\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\nset_seed(42)\n\n# 2. Carga y estructuración de dataset.csv (Formato Melt -> Matrices del Grafo)\ndf = pd.read_csv(\'/kaggle/input/datasets/nico14h/metagenomica/dataset.csv\', sep=None, engine= \'python\')\n# Mapeo ordenado de las 153 bacterias y los 121 pacientes\nbact_map = df[[\'bacteria_id\', \'Bacteria_Limpia\']].drop_duplicates().sort_values(\'bacteria_id\')\nnombres_bacterias = bact_map[\'Bacteria_Limpia\'].values\n\npat_map = df[[\'paciente_id\', \'Paciente\', \'Diagnosticos\']].drop_duplicates().sort_values(\'paciente_id\')\ny_all = pat_map[\'Diagnosticos\'].values  # 1 = TEA (71 pacientes), 0 = HC (50 pacientes)\n\n# Matriz de Abundancia CLR (121 x 153) y Máscara de Presencia de Aristas Bipartitas\npivot_clr = df.pivot(index=\'paciente_id\', columns=\'bacteria_id\', values=\'Abundanc

In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    recall_score, confusion_matrix
)

# =====================================================================
# 1. HIPERPARÁMETROS CONFIGURABLES DE LA ARQUITECTURA Y ENTRENAMIENTO
# =====================================================================
IN_CHANNELS = 8           # 8 características topológicas/ecológicas por bacteria (sin etiquetas)
HIDDEN_CHANNELS = 32      # <--- Neuronas por capa oculta (ancho del embedding: ej. 16, 32, 64)
OUT_CHANNELS = 2          # <--- Canales de salida (2: Canal 0 = HC, Canal 1 = TEA)
NUM_LAYERS = 2            # <--- Cantidad de capas ocultas GATv2 apiladas (profundidad: ej. 1, 2)
DROPOUT = 0.4            # 25% de apagado aleatorio para regularización
EPOCHS = 100              # Épocas de entrenamiento por Fold
LR = 0.005                # Tasa de aprendizaje (AdamW)
WEIGHT_DECAY = 2e-2       # Regularización L2

# Configuración del desbalance en Entrenamiento (56 TEA vs 40 HC por Fold)
COMPENSAR_56_VS_40 = True # True: multiplica por (56/40 = 1.40) y encima suma EXTRA_PESO_HC
EXTRA_PESO_HC = 0.07      # <--- Entre 0.04 (4%) y 0.10 (10%) de peso superior para HC sobre TEA

# Umbral estadístico para crear Cables Rojos (TEA) y Cables Verdes (HC)
UMBRAL_CORRELACION = 0.35 # |r| >= 0.35 (p < 0.01 en TEA N=56, p < 0.03 en HC N=40)


# =====================================================================
# 2. CARGA DEL DATASET (SIN SEMILLA FIJA -> CAMBIA EN CADA EJECUCIÓN)
# =====================================================================
# Ajustá la ruta si corrés en Kaggle ('/kaggle/working/dataset.csv') o local ('dataset.csv')
RUTA_DATASET = '/kaggle/input/datasets/nico14h/metagenomica/dataset.csv'
df = pd.read_csv(RUTA_DATASET, sep=None, engine='python')

# Verificación de seguridad de la limpieza (Celda 5) y reindexación contigua (0 a N-1)
generos_a_eliminar = [
    'Ralstonia', 'Burkholderia', 'Cupriavidus', 'Methylobacterium',
    'Bradyrhizobium', 'Rhizobium', 'Sphingomonas', 'Novosphingobium',
    'Acetobacter', 'Gluconobacter', 'Oenococcus', 'Cutibacterium',
    'uncultured', 'Desconocida'
]
df = df[~df['Bacteria_Limpia'].isin(generos_a_eliminar)].copy()
df.dropna(subset=['AbundanciaTaxonomica'], inplace=True)

df['paciente_id'] = pd.factorize(df['Paciente'])[0]
df['bacteria_id'] = pd.factorize(df['Bacteria_Limpia'])[0]

bact_map = df[['bacteria_id', 'Bacteria_Limpia']].drop_duplicates().sort_values('bacteria_id')
nombres_bacterias = bact_map['Bacteria_Limpia'].values
num_bacterias = len(nombres_bacterias)

pat_map = df[['paciente_id', 'Paciente', 'Diagnosticos']].drop_duplicates().sort_values('paciente_id')
y_all = pat_map['Diagnosticos'].values  # 1 = TEA (71 pacientes), 0 = HC (50 pacientes)
num_pacientes = len(y_all)

# Grilla de Pacientes x Bacterias conservando NaN donde la bacteria está AUSENTE
pivot_rclr = df.pivot(index='paciente_id', columns='bacteria_id', values='AbundanciaTaxonomica')
RCLR_raw = pivot_rclr.values                      # (121, 152) con NaN en las ausencias reales
M_all = (~np.isnan(RCLR_raw)).astype(np.float32)  # 1.0 donde existe Cable Morado activo, 0.0 si está ausente

print(f"Dataset cargado: {num_pacientes} Pacientes ({int((y_all==1).sum())} TEA, {int((y_all==0).sum())} HC) | "
      f"{num_bacterias} Bacterias limpias | {int(M_all.sum())} Cables Morados activos.")


# =====================================================================
# 3. ARQUITECTURA GATv2 HETEROGÉNEA MULTIRELACIONAL 100% NO DIRIGIDA
# =====================================================================
class SymmetricGATv2BacteriaLayer(nn.Module):
    """
    Cabezal GATv2 NO DIRIGIDO (Simétrico) para un color de cable Bacteria <-> Bacteria
    (Cables Rojos para TEA o Cables Verdes para HC).
    Distingue correlaciones positivas (simbiosis) de negativas (competencia).
    """
    def __init__(self, in_channels, out_channels, dropout=0.25, alpha=0.2):
        super().__init__()
        self.W_node = nn.Linear(in_channels, out_channels, bias=False)
        self.W_edge = nn.Linear(2, out_channels, bias=False)  # [|r|, r_con_signo]
        self.a = nn.Linear(out_channels, 1, bias=False)

        nn.init.xavier_uniform_(self.W_node.weight, gain=1.414)
        nn.init.xavier_uniform_(self.W_edge.weight, gain=1.414)
        nn.init.xavier_uniform_(self.a.weight, gain=1.414)

        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h_bact, adj_matrix):
        B = adj_matrix.size(0)
        eye = torch.eye(B, device=adj_matrix.device)
        adj_with_self = adj_matrix + eye
        mask = (torch.abs(adj_with_self) > 0)  # (B, B)

        # Atributos de la arista Bacteria <-> Bacteria: magnitud |r| y valor con signo r
        edge_feat = torch.stack([torch.abs(adj_with_self), adj_with_self], dim=-1)  # (B, B, 2)
        We = self.W_edge(edge_feat)  # (B, B, H)

        Wh = self.W_node(h_bact)     # (B, H)
        # Ecuación GATv2 simétrica: Wh_i + Wh_j + We_ij -> LeakyReLU -> vector de atención 'a'
        sum_sym = Wh.unsqueeze(1) + Wh.unsqueeze(0) + We  # (B, B, H)
        e = self.a(self.leakyrelu(sum_sym)).squeeze(-1)   # (B, B)
        e_masked = e.masked_fill(~mask, -1e9)

        attention = F.softmax(e_masked, dim=-1)
        attention_drop = self.dropout(attention)

        # Paso de mensajes modulado por el signo y fuerza de la correlación
        h_prime = torch.matmul(attention_drop * adj_with_self, Wh)  # (B, H)
        return F.elu(h_prime), attention


class UndirectedHeteroMicrobiomeGATv2(nn.Module):
    """
    Grafo Heterogéneo No Dirigido en 3 Fases por Capa:
      - Nodos Paciente arrancan VACÍOS (h_pat = None).
      - Nodos Bacteria arrancan con 8 características topológicas de sus cables.
      - Fase 1 (B <-> B): Cabezal Rojo (TEA) y Cabezal Verde (HC) actualizan las bacterias.
      - Fase 2 (B -> P): Por los Cables Morados activos, las bacterias construyen el estado del Paciente.
      - Fase 3 (P -> B): Por los mismos Cables Morados, los pacientes de entrenamiento retroalimentan a las bacterias.
    """
    def __init__(self, in_channels=8, hidden_channels=32, out_channels=2, num_layers=2, dropout=0.4):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_channels = hidden_channels
        self.dropout = dropout

        # Proyección inicial de las 8 características topológicas de las bacterias
        self.bact_input_proj = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.LayerNorm(hidden_channels),
            nn.ELU()
        )

        # Listas dinámicas de capas según NUM_LAYERS
        self.red_heads = nn.ModuleList()
        self.green_heads = nn.ModuleList()
        self.bact_fusions = nn.ModuleList()

        self.W_bact_purple = nn.ModuleList()
        self.W_pat_purple = nn.ModuleList()
        self.W_edge_purple = nn.ModuleList()
        self.att_purple = nn.ModuleList()
        self.msg_b2p = nn.ModuleList()
        self.msg_p2b = nn.ModuleList()
        self.pat_norms = nn.ModuleList()
        self.bact_norms = nn.ModuleList()

        for _ in range(num_layers):
            self.red_heads.append(SymmetricGATv2BacteriaLayer(hidden_channels, hidden_channels, dropout=dropout))
            self.green_heads.append(SymmetricGATv2BacteriaLayer(hidden_channels, hidden_channels, dropout=dropout))

            # Fusión ecológica: [Estado_Propio, Canal_Rojo, Canal_Verde, Contraste(Rojo - Verde)]
            self.bact_fusions.append(
                nn.Sequential(
                    nn.Linear(hidden_channels * 4, hidden_channels),
                    nn.LayerNorm(hidden_channels),
                    nn.ELU()
                )
            )

            # Parámetros GATv2 para los Cables Morados No Dirigidos (Paciente <-> Bacteria)
            self.W_bact_purple.append(nn.Linear(hidden_channels, hidden_channels, bias=False))
            self.W_pat_purple.append(nn.Linear(hidden_channels, hidden_channels, bias=False))
            self.W_edge_purple.append(nn.Linear(3, hidden_channels, bias=False))
            self.att_purple.append(nn.Linear(hidden_channels, 1, bias=False))

            self.msg_b2p.append(nn.Linear(hidden_channels + 3, hidden_channels))
            self.msg_p2b.append(nn.Linear(hidden_channels + 3, hidden_channels))
            self.pat_norms.append(nn.LayerNorm(hidden_channels))
            self.bact_norms.append(nn.LayerNorm(hidden_channels))

        # Cabezal Clasificador Final hacia OUT_CHANNELS (Canal 0 = Logit_HC, Canal 1 = Logit_TEA)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_channels),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, out_channels)
        )

    def forward(self, bact_features, adj_red, adj_green, edge_feats_morados, train_mask_nodes=None):
        P, B, _ = edge_feats_morados.shape
        mask_morado = edge_feats_morados[:, :, 2] > 0  # (121, 152) True solo donde existe Cable Morado
        peso_positivo = edge_feats_morados[:, :, 0:1]  # (121, 152, 1) Carga bacteriana sobre el piso (> 0)

        # Estado inicial de los nodos Bacteria (152, H)
        h_bact = self.bact_input_proj(bact_features)
        # Los nodos Paciente arrancan completamente VACÍOS en el instante cero
        h_pat = None

        for layer in range(self.num_layers):
            # -----------------------------------------------------------------
            # FASE 1: Interacción Ecológica NO DIRIGIDA Bacteria <-> Bacteria
            # -----------------------------------------------------------------
            h_red, _ = self.red_heads[layer](h_bact, adj_red)      # Cabezal Rojo (TEA)
            h_green, _ = self.green_heads[layer](h_bact, adj_green) # Cabezal Verde (HC)
            h_contrast = h_red - h_green

            h_bact = self.bact_fusions[layer](
                torch.cat([h_bact, h_red, h_green, h_contrast], dim=-1)
            )  # (152, H)

            # -----------------------------------------------------------------
            # FASE 2: Colonización Bacteria -> Paciente (Cables Morados)
            # -----------------------------------------------------------------
            Wh_b = self.W_bact_purple[layer](h_bact).unsqueeze(0).expand(P, B, -1)  # (121, 152, H)
            We_p = self.W_edge_purple[layer](edge_feats_morados)                    # (121, 152, H)

            if h_pat is None:
                # En la Capa 1 el Paciente está vacío: recibe pura información desde sus bacterias y cables morados
                energia_b2p = F.leaky_relu(Wh_b + We_p, negative_slope=0.2)
            else:
                # En capas siguientes el Paciente ya tiene estado: energía simétrica completa
                Wh_p = self.W_pat_purple[layer](h_pat).unsqueeze(1).expand(P, B, -1)
                energia_b2p = F.leaky_relu(Wh_b + Wh_p + We_p, negative_slope=0.2)

            scores_b2p = self.att_purple[layer](energia_b2p).squeeze(-1)            # (121, 152)
            # Enmascarar con -infinito las bacterias ausentes en cada niño
            scores_b2p_masked = scores_b2p.masked_fill(~mask_morado, -1e9)

            alpha_b2p = F.softmax(scores_b2p_masked, dim=1) * mask_morado.float()   # (121, 152)

            h_bact_exp = h_bact.unsqueeze(0).expand(P, B, -1)
            msg_in_b2p = torch.cat([h_bact_exp * peso_positivo, edge_feats_morados], dim=-1)
            mensajes_b2p = F.elu(self.msg_b2p[layer](msg_in_b2p))                   # (121, 152, H)
            h_pat_agregado = torch.sum(alpha_b2p.unsqueeze(-1) * mensajes_b2p, dim=1)  # (121, H)

            if h_pat is None:
                h_pat = self.pat_norms[layer](h_pat_agregado)
            else:
                h_pat = self.pat_norms[layer](
                    h_pat + F.dropout(h_pat_agregado, p=self.dropout, training=self.training)
                )

            # -----------------------------------------------------------------
            # FASE 3: Retroalimentación NO DIRIGIDA Paciente -> Bacteria
            # -----------------------------------------------------------------
            Wh_p_back = self.W_pat_purple[layer](h_pat).unsqueeze(1).expand(P, B, -1)
            energia_p2b = F.leaky_relu(Wh_b + Wh_p_back + We_p, negative_slope=0.2)
            scores_p2b = self.att_purple[layer](energia_p2b).squeeze(-1)

            # Para evitar fuga de datos, solo los pacientes de entrenamiento retroalimentan a las bacterias globales
            if train_mask_nodes is not None:
                mask_p2b = mask_morado & train_mask_nodes.unsqueeze(1)
            else:
                mask_p2b = mask_morado

            scores_p2b_masked = scores_p2b.masked_fill(~mask_p2b, -1e9)
            alpha_p2b = F.softmax(scores_p2b_masked, dim=0) * mask_p2b.float()      # (121, 152)

            msg_in_p2b = torch.cat([Wh_p_back * peso_positivo, edge_feats_morados], dim=-1)
            mensajes_p2b = F.elu(self.msg_p2b[layer](msg_in_p2b))
            h_bact_from_pat = torch.sum(alpha_p2b.unsqueeze(-1) * mensajes_p2b, dim=0)  # (152, H)

            h_bact = self.bact_norms[layer](
                h_bact + F.dropout(h_bact_from_pat, p=self.dropout, training=self.training)
            )

        logits = self.classifier(h_pat)  # (121, OUT_CHANNELS) -> [Logit_HC, Logit_TEA]
        return logits, alpha_b2p, alpha_p2b, h_red, h_green


# =====================================================================
# 4. CONSTRUCTOR DEL GRAFO POR FOLD (PISO rCLR + CERO DATA LEAKAGE)
# =====================================================================
def build_fold_hetero_graph(RCLR_raw, M_all, y_all, train_idx, corr_thresh=0.35):
    RCLR_train = RCLR_raw[train_idx]
    M_train = M_all[train_idx]
    y_train = y_all[train_idx]

    # 1. Piso de detección biológico calculado SOLO en train_idx (evita el error de fillna(0.0))
    min_rclr_train = float(np.nanmin(RCLR_train))
    piso_deteccion = min_rclr_train - 0.5  # ~ -4.93: Ausente < Presente baja < Presente media < Presente alta

    X_floor_all = np.where(np.isnan(RCLR_raw), piso_deteccion, RCLR_raw)
    X_floor_train = X_floor_all[train_idx]

    X_train_tea = X_floor_train[y_train == 1]  # Exactamente 56 pacientes TEA
    X_train_hc = X_floor_train[y_train == 0]   # Exactamente 40 pacientes HC

    # 2. Matrices de correlación simétricas para Cables Rojos (TEA) y Cables Verdes (HC)
    corr_tea = np.nan_to_num(np.corrcoef(X_train_tea, rowvar=False), 0.0)
    corr_hc = np.nan_to_num(np.corrcoef(X_train_hc, rowvar=False), 0.0)
    np.fill_diagonal(corr_tea, 0.0)
    np.fill_diagonal(corr_hc, 0.0)

    adj_red = np.where(np.abs(corr_tea) >= corr_thresh, corr_tea, 0.0)
    adj_green = np.where(np.abs(corr_hc) >= corr_thresh, corr_hc, 0.0)

    info_cables = {
        'rojos_tea': int((np.abs(adj_red) > 0).sum() // 2),
        'verdes_hc': int((np.abs(adj_green) > 0).sum() // 2),
        'doble_rojo_verde': int(((np.abs(adj_red) > 0) & (np.abs(adj_green) > 0)).sum() // 2),
        'morados_train': int(M_train.sum()),
        'morados_totales': int(M_all.sum()),
        'piso_rclr': piso_deteccion
    }

    # 3. Ocho (8) Características Puramente Topológicas y Ecológicas de cada Bacteria
    deg_red = (np.abs(adj_red) > 0).mean(axis=0)
    deg_green = (np.abs(adj_green) > 0).mean(axis=0)
    simbiosis_red = np.maximum(0.0, adj_red).mean(axis=0)
    competencia_red = np.maximum(0.0, -adj_red).mean(axis=0)
    simbiosis_green = np.maximum(0.0, adj_green).mean(axis=0)
    competencia_green = np.maximum(0.0, -adj_green).mean(axis=0)
    recableado_l2 = np.linalg.norm(adj_red - adj_green, axis=1) / np.sqrt(adj_red.shape[0])
    prevalencia_global_train = M_train.mean(axis=0)

    bact_features = np.stack([
        deg_red, deg_green,
        simbiosis_red, competencia_red,
        simbiosis_green, competencia_green,
        recableado_l2, prevalencia_global_train
    ], axis=1)

    bf_mean = bact_features.mean(axis=0, keepdims=True)
    bf_std = bact_features.std(axis=0, keepdims=True) + 1e-6
    bact_features_norm = (bact_features - bf_mean) / bf_std

    # 4. Atributos de los Cables Morados (Paciente <-> Bacteria)
    rango_train = (float(np.nanmax(RCLR_train)) - piso_deteccion) + 1e-6
    peso_positivo = np.where(M_all > 0, (X_floor_all - piso_deteccion) / rango_train, 0.0)

    mean_tr = X_floor_train.mean(axis=0, keepdims=True)
    std_tr = X_floor_train.std(axis=0, keepdims=True) + 1e-6
    z_score_arista = np.where(M_all > 0, (X_floor_all - mean_tr) / std_tr, 0.0)

    edge_feats_morados = np.stack([peso_positivo, z_score_arista, M_all], axis=-1)

    return (
        torch.tensor(bact_features_norm, dtype=torch.float32),
        torch.tensor(adj_red, dtype=torch.float32),
        torch.tensor(adj_green, dtype=torch.float32),
        torch.tensor(edge_feats_morados, dtype=torch.float32),
        X_floor_all,
        info_cables
    )


# =====================================================================
# 5. VALIDACIÓN CRUZADA (5-FOLD) Y EXTRACCIÓN DE BIOMARCADORES TEA Y HC
# =====================================================================
skf = StratifiedKFold(n_splits=5, shuffle=True)
metricas_folds = []

# Acumuladores específicos para clasificar TEA (Clase 1)
atencion_tea_acum = np.zeros(num_bacterias)
impulso_grad_tea_acum = np.zeros(num_bacterias)
red_roja_acum = np.zeros(num_bacterias)

# Acumuladores específicos para clasificar Controles Sanos HC (Clase 0)
atencion_hc_acum = np.zeros(num_bacterias)
impulso_grad_hc_acum = np.zeros(num_bacterias)
red_verde_acum = np.zeros(num_bacterias)

# Acumulador de diferencia biológica real (corregida con piso de detección)
delta_rclr_real_acum = np.zeros(num_bacterias)

print("\n=== ENTRENANDO GATv2 HETEROGÉNEO NO DIRIGIDO (56 TEA vs 40 HC POR FOLD) ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(M_all, y_all)):
    # Ajuste exacto para garantizar 56 TEA y 40 HC en el entrenamiento de cada Fold (96 pacientes)
    tea_en_train = train_idx[y_all[train_idx] == 1]
    if len(tea_en_train) > 56:
        sobrante_tea = np.random.choice(tea_en_train, size=len(tea_en_train) - 56, replace=False)
        train_idx = np.setdiff1d(train_idx, sobrante_tea)
        val_idx = np.concatenate([val_idx, sobrante_tea])

    n_tea_train = int((y_all[train_idx] == 1).sum())  # Siempre 56
    n_hc_train = int((y_all[train_idx] == 0).sum())   # Siempre 40
    n_tea_val = int((y_all[val_idx] == 1).sum())      # Siempre 15
    n_hc_val = int((y_all[val_idx] == 0).sum())       # Siempre 10

    train_mask_bool = torch.zeros(num_pacientes, dtype=torch.bool)
    train_mask_bool[train_idx] = True

    bact_feat, adj_red, adj_green, edge_morados, X_floor_all, info_cables = build_fold_hetero_graph(
        RCLR_raw, M_all, y_all, train_idx, corr_thresh=UMBRAL_CORRELACION
    )
    y_tensor = torch.tensor(y_all, dtype=torch.long)

    # Se instancia un modelo nuevo desde cero con pesos aleatorios en cada Fold
    model = UndirectedHeteroMicrobiomeGATv2(
        in_channels=IN_CHANNELS,
        hidden_channels=HIDDEN_CHANNELS,
        out_channels=OUT_CHANNELS,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Ponderación de clases: Equilibrio 56/40 (1.40) + recargo extra del 4% al 10% para HC
    peso_tea = 1.0
    if COMPENSAR_56_VS_40:
        peso_hc = (n_tea_train / n_hc_train) * (1.0 + EXTRA_PESO_HC)
    else:
        peso_hc = 1.0 + EXTRA_PESO_HC

    pesos_clases = torch.tensor([peso_hc, peso_tea], dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=pesos_clases)

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits, _, _, _, _ = model(bact_feat, adj_red, adj_green, edge_morados, train_mask_nodes=train_mask_bool)
        loss = criterion(logits[train_idx], y_tensor[train_idx])
        loss.backward()
        optimizer.step()

    # Evaluación en el conjunto de Validación (val_idx: 15 TEA y 10 HC)
    model.eval()
    edge_morados_grad = edge_morados.clone().detach().requires_grad_(True)
    logits, alpha_b2p, alpha_p2b, h_red, h_green = model(
        bact_feat, adj_red, adj_green, edge_morados_grad, train_mask_nodes=train_mask_bool
    )

    # Gradiente del margen diagnóstico (Logit_TEA - Logit_HC) respecto al peso del Cable Morado
    margen_tea = (logits[:, 1] - logits[:, 0]).sum()
    margen_tea.backward()

    with torch.no_grad():
        probs_tea = F.softmax(logits[val_idx], dim=1)[:, 1].numpy()
        preds = (probs_tea >= 0.5).astype(int)
        y_val = y_all[val_idx]

        acc = accuracy_score(y_val, preds)
        auc = roc_auc_score(y_val, probs_tea)
        f1 = f1_score(y_val, preds)
        sens = recall_score(y_val, preds)
        tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
        spec = tn / (tn + fp)

        metricas_folds.append([fold + 1, acc, auc, f1, sens, spec])

        print(f"\n--- FOLD {fold+1} ---")
        print(f"Pacientes -> Train: {len(train_idx)} ({n_tea_train} TEA, {n_hc_train} HC) | "
              f"Val: {len(val_idx)} ({n_tea_val} TEA, {n_hc_val} HC) | "
              f"Pesos Loss -> HC: {peso_hc:.3f}, TEA: {peso_tea:.3f}")
        print(f"Cables    -> Rojos (TEA B<->B): {info_cables['rojos_tea']} | "
              f"Verdes (HC B<->B): {info_cables['verdes_hc']} | "
              f"Ambos (Rojo+Verde): {info_cables['doble_rojo_verde']} | "
              f"Morados (P<->B): {info_cables['morados_train']} en Train ({info_cables['morados_totales']} Total)")
        print(f"Métricas  -> Acc: {acc:.4f} | ROC-AUC: {auc:.4f} | F1: {f1:.4f} | "
              f"Sens(TEA): {sens:.4f} | Spec(HC): {spec:.4f}")

        tea_val_nodes = val_idx[y_all[val_idx] == 1]
        hc_val_nodes = val_idx[y_all[val_idx] == 0]

        # 1. Atención en los Cables Morados de pacientes TEA vs pacientes HC
        att_tea = alpha_b2p[tea_val_nodes].mean(dim=0).numpy()
        att_hc = alpha_b2p[hc_val_nodes].mean(dim=0).numpy()

        # 2. Impulso Direccional (Peso_Cable * Gradiente con signo hacia cada patología)
        grad_peso = edge_morados_grad.grad[:, :, 0]
        peso_val = edge_morados[:, :, 0]

        impulso_tea = F.relu(peso_val[tea_val_nodes] * grad_peso[tea_val_nodes]).mean(dim=0).numpy()
        impulso_hc = F.relu(peso_val[hc_val_nodes] * (-grad_peso[hc_val_nodes])).mean(dim=0).numpy()

        atencion_tea_acum += att_tea / 5.0
        impulso_grad_tea_acum += impulso_tea / 5.0
        red_roja_acum += torch.norm(h_red, p=2, dim=1).numpy() / 5.0

        atencion_hc_acum += att_hc / 5.0
        impulso_grad_hc_acum += impulso_hc / 5.0
        red_verde_acum += torch.norm(h_green, p=2, dim=1).numpy() / 5.0

        # Diferencia biológica real calculada con el piso de detección (sin distorsión de fillna(0.0))
        delta_fold = X_floor_all[y_all == 1].mean(axis=0) - X_floor_all[y_all == 0].mean(axis=0)
        delta_rclr_real_acum += delta_fold / 5.0

df_metricas = pd.DataFrame(
    metricas_folds,
    columns=['Fold', 'Accuracy', 'ROC_AUC', 'F1_Score', 'Sensibilidad_TEA', 'Especificidad_HC']
)
print("\n=== PROMEDIO DE VALIDACIÓN CRUZADA (5-FOLD) ===")
print(df_metricas[['Accuracy', 'ROC_AUC', 'F1_Score', 'Sensibilidad_TEA', 'Especificidad_HC']].mean().to_string())


# =====================================================================
# 6. TABLAS FINALES: TOP BACTERIAS PARA TEA Y TOP BACTERIAS PARA HC
# =====================================================================
prev_tea_global = M_all[y_all == 1].mean(axis=0)
prev_hc_global = M_all[y_all == 0].mean(axis=0)

def norm_minmax(v):
    return (v - v.min()) / (v.max() - v.min() + 1e-8)

# Puntaje integral para clasificar TEA (Impulso hacia TEA + Atención en TEA + Fuerza en Red Roja)
score_gat_tea = (
    0.40 * norm_minmax(impulso_grad_tea_acum) +
    0.35 * norm_minmax(atencion_tea_acum) +
    0.25 * norm_minmax(red_roja_acum)
)

# Puntaje integral para clasificar Controles Sanos HC (Impulso hacia HC + Atención en HC + Fuerza en Red Verde)
score_gat_hc = (
    0.40 * norm_minmax(impulso_grad_hc_acum) +
    0.35 * norm_minmax(atencion_hc_acum) +
    0.25 * norm_minmax(red_verde_acum)
)

df_biomarcadores_tea = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Score_GAT_TEA': score_gat_tea,
    'Atencion_Morada_TEA': atencion_tea_acum,
    'Impulso_Gradiente_TEA': impulso_grad_tea_acum,
    'Peso_Red_Roja': red_roja_acum,
    'Prevalencia_TEA': prev_tea_global,
    'Prevalencia_HC': prev_hc_global,
    'Delta_rCLR_Real (TEA-HC)': delta_rclr_real_acum,
    'Rol_Clinico_en_TEA': np.where(
        delta_rclr_real_acum > 0,
        'Aumentada en TEA (Sobrecrecimiento / Red Roja)',
        'Disminuida en TEA (Depleción)'
    )
}).sort_values(by='Score_GAT_TEA', ascending=False)

df_biomarcadores_hc = pd.DataFrame({
    'Bacteria': nombres_bacterias,
    'Score_GAT_HC': score_gat_hc,
    'Atencion_Morada_HC': atencion_hc_acum,
    'Impulso_Gradiente_HC': impulso_grad_hc_acum,
    'Peso_Red_Verde': red_verde_acum,
    'Prevalencia_HC': prev_hc_global,
    'Prevalencia_TEA': prev_tea_global,
    'Delta_rCLR_Real (HC-TEA)': -delta_rclr_real_acum,
    'Rol_Clinico_en_HC': np.where(
        delta_rclr_real_acum < 0,
        'Aumentada en HC (Protectora / Red Verde)',
        'Baja en HC (Control de Patobiontes)'
    )
}).sort_values(by='Score_GAT_HC', ascending=False)

print("\n=== TOP 20 BACTERIAS MÁS IMPORTANTES PARA CLASIFICAR NIÑOS CON TEA ===")
print(df_biomarcadores_tea.head(20).to_string(index=False))

print("\n=== TOP 20 BACTERIAS MÁS IMPORTANTES PARA CLASIFICAR CONTROLES SANOS (HC) ===")
print(df_biomarcadores_hc.head(20).to_string(index=False))

Dataset cargado: 121 Pacientes (71 TEA, 50 HC) | 152 Bacterias limpias | 10679 Cables Morados activos.

=== ENTRENANDO GATv2 HETEROGÉNEO NO DIRIGIDO (56 TEA vs 40 HC POR FOLD) ===

--- FOLD 1 ---
Pacientes -> Train: 96 (56 TEA, 40 HC) | Val: 25 (15 TEA, 10 HC) | Pesos Loss -> HC: 1.498, TEA: 1.000
Cables    -> Rojos (TEA B<->B): 1071 | Verdes (HC B<->B): 2211 | Ambos (Rojo+Verde): 420 | Morados (P<->B): 8512 en Train (10679 Total)
Métricas  -> Acc: 0.6400 | ROC-AUC: 0.5933 | F1: 0.6897 | Sens(TEA): 0.6667 | Spec(HC): 0.6000

--- FOLD 2 ---
Pacientes -> Train: 96 (56 TEA, 40 HC) | Val: 25 (15 TEA, 10 HC) | Pesos Loss -> HC: 1.498, TEA: 1.000
Cables    -> Rojos (TEA B<->B): 1012 | Verdes (HC B<->B): 2190 | Ambos (Rojo+Verde): 517 | Morados (P<->B): 8409 en Train (10679 Total)
Métricas  -> Acc: 0.6800 | ROC-AUC: 0.7400 | F1: 0.6923 | Sens(TEA): 0.6000 | Spec(HC): 0.8000

--- FOLD 3 ---
Pacientes -> Train: 96 (56 TEA, 40 HC) | Val: 25 (15 TEA, 10 HC) | Pesos Loss -> HC: 1.498, TEA: 1.000
C